In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

from google.colab import drive
drive.mount('/content/drive')
# Filepath for saving to Drive, change to your own folder setup:

# Tim drive paths
DRIVE_PATH = '/content/drive/MyDrive/CrispHMM/hmmlearn_and_other/multichrresults'
MERGED_DIR  = '/content/drive/MyDrive/CrispHMM/sgRNA'

# David drive paths
#DRIVE_PATH = '/content/drive/MyDrive/Spring 2026/CBMF W4761/Project/CrispHMM/hmmlearn_and_other/chr11results'
#DATA_PATH  = '/content/drive/MyDrive/Spring 2026/CBMF W4761/Project/CrispHMM/sgRNA/HeLa_chr11_merged.csv'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install pyro-ppl

In [ ]:
import argparse
import logging
import sys
import seaborn as sn
import torch
import torch.nn as nn
from torch.distributions import constraints

import pyro
import pyro.contrib.examples.polyphonic_data_loader as poly
import pyro.distributions as dist
from pyro import poutine
from pyro.infer import SVI, JitTraceEnum_ELBO, JitTrace_ELBO, TraceEnum_ELBO, Trace_ELBO, TraceTMC_ELBO
from pyro.infer.autoguide import AutoDelta
from pyro.ops.indexing import Vindex
from pyro.optim import Adam
from pyro.util import ignore_jit_warnings
import pandas as pd
import numpy as np

import types

import matplotlib.pyplot as plt
import seaborn as sns

logging.basicConfig(format="%(relativeCreated) 9d %(message)s", level=logging.DEBUG)

torch.manual_seed(32)
pyro.set_rng_seed(32)

In [ ]:
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
    print("Using CPU, not Nvidia CUDA or Apple MPS")
print(DEVICE)
torch.set_default_device(DEVICE)

cuda


In [ ]:
args = types.SimpleNamespace(
    hidden_dim=11,       # number of chromatin states
    batch_size=2,        # minibatch
    learning_rate=0.02,
    cuda=True,
    jit=False,
    num_cont_dim = 1,
    num_bin_dim = 9,
    num_eff_features = 3,
)

In [ ]:

MARK_COLS     = ['CTCF','H3K27ac','H3K27me3','H3K36me3',
                 'H3K4me1','H3K4me2','H3K4me3','H3K9ac','H4K20me1']
EFF_FEAT_COLS = ['gc_content', 'mfe_kcal', 'tm_celsius']
CONT_COL      = ['Normalized efficacy']

TEST_CHROMS       = ['chr13', 'chr21', 'chr22']   # ~265 sgRNAs, never touch until final eval
VALIDATION_CHROMS = ['chr18', 'chr20']             # ~257 sgRNAs, monitor in case of overfitting
# Everything else is training

# Store held-out ground truth before masking
test_true = {}
val_true  = {}

# Find all merged files — skip chrM
files = sorted([
    f for f in os.listdir(MERGED_DIR)
    if f.endswith('_merged.csv') and 'chrM' not in f
])
print(f"Found {len(files)} chromosome files")

sequences_list    = []
eff_features_list = []
lengths_list      = []
chrom_names       = []

for filename in files:
    chrom = filename.replace('HeLa_', '').replace('_merged.csv', '')
    filepath = os.path.join(MERGED_DIR, filename)
    df = pd.read_csv(filepath)

    # Store and mask held-out efficiency values
    if chrom in TEST_CHROMS:
        test_true[chrom] = df[df['Normalized efficacy'].notna()][
            ['bin_index', 'Normalized efficacy']].copy()
        df['Normalized efficacy'] = np.nan
        print(f"  {chrom}: TEST  — {len(test_true[chrom])} sgRNAs masked")

    elif chrom in VALIDATION_CHROMS:
        val_true[chrom] = df[df['Normalized efficacy'].notna()][
            ['bin_index', 'Normalized efficacy']].copy()
        df['Normalized efficacy'] = np.nan
        print(f"  {chrom}: VAL   — {len(val_true[chrom])} sgRNAs masked")

    else:
        n_sgrna = df['Normalized efficacy'].notna().sum()
        print(f"  {chrom}: TRAIN — {n_sgrna} sgRNAs")

    # Build tensors
    marks  = torch.tensor(df[MARK_COLS].values,    dtype=torch.float32)
    cont   = torch.tensor(df[CONT_COL].values,     dtype=torch.float32)
    feats  = torch.tensor(df[EFF_FEAT_COLS].values, dtype=torch.float32)
    seq    = torch.cat([marks, cont], dim=-1)

    sequences_list.append(seq)
    eff_features_list.append(feats)
    lengths_list.append(len(df))
    chrom_names.append(chrom)

# Sort by length ascending before stacking
# This way short chromosomes tend to batch together
sorted_indices = sorted(range(len(lengths_list)), key=lambda i: lengths_list[i])

sequences_list    = [sequences_list[i] for i in sorted_indices]
eff_features_list = [eff_features_list[i] for i in sorted_indices]
lengths_list      = [lengths_list[i] for i in sorted_indices]
chrom_names       = [chrom_names[i] for i in sorted_indices]


# Pad all chromosomes to same length for batching
max_len = max(lengths_list)

def pad(t, max_len):
    pad_size = max_len - t.shape[0]
    if pad_size == 0:
        return t
    return torch.cat([t, torch.zeros(pad_size, t.shape[1])], dim=0)

sequences_0  = torch.stack([pad(s, max_len) for s in sequences_list])    # [N, max_T, 10]
eff_features = torch.stack([pad(f, max_len) for f in eff_features_list]) # [N, max_T, 3]
lengths_0    = torch.tensor(lengths_list, dtype=torch.long)               # [N]

print("Chromosomes sorted by length:")
for name, length in zip(chrom_names, lengths_list):
    print(f"  {name}: {length:,} bins")

N = len(sequences_list)

# Count training sgRNAs from already-loaded data
train_sgrna_count = 0
for i, chrom in enumerate(chrom_names):
    if chrom not in TEST_CHROMS + VALIDATION_CHROMS:
        # Count non-NaN efficacy values in the continuous column (index 9)
        train_sgrna_count += (~torch.isnan(sequences_list[i][:, 9])).sum().item()

print(f"\n=== Data Summary ===")
print(f"Chromosomes loaded: {N}")
print(f"sequences:    {sequences_0.shape}")
print(f"eff_features: {eff_features.shape}")
print(f"lengths:      min={lengths_0.min()}, max={lengths_0.max()}")
print(f"Train sgRNAs: {train_sgrna_count}")
print(f"Val sgRNAs:   {sum(len(v) for v in val_true.values())}")
print(f"Test sgRNAs:  {sum(len(t) for t in test_true.values())}")


Found 23 chromosome files
  chr10: TRAIN — 197 sgRNAs
  chr11: TRAIN — 410 sgRNAs
  chr12: TRAIN — 314 sgRNAs
  chr13: TEST  — 80 sgRNAs masked
  chr14: TRAIN — 245 sgRNAs
  chr15: TRAIN — 179 sgRNAs
  chr16: TRAIN — 298 sgRNAs
  chr17: TRAIN — 461 sgRNAs
  chr18: VAL   — 73 sgRNAs masked
  chr19: TRAIN — 386 sgRNAs
  chr1: TRAIN — 546 sgRNAs
  chr20: VAL   — 184 sgRNAs masked
  chr21: TEST  — 85 sgRNAs masked
  chr22: TEST  — 100 sgRNAs masked
  chr2: TRAIN — 371 sgRNAs
  chr3: TRAIN — 299 sgRNAs
  chr4: TRAIN — 159 sgRNAs
  chr5: TRAIN — 224 sgRNAs
  chr6: TRAIN — 247 sgRNAs
  chr7: TRAIN — 224 sgRNAs
  chr8: TRAIN — 169 sgRNAs
  chr9: TRAIN — 251 sgRNAs
  chrX: TRAIN — 154 sgRNAs
Chromosomes sorted by length:
  chr21: 240,649 bins
  chr22: 256,522 bins
  chr19: 295,644 bins
  chr20: 315,127 bins
  chr18: 390,386 bins
  chr17: 405,976 bins
  chr16: 451,773 bins
  chr15: 512,656 bins
  chr14: 536,747 bins
  chr13: 575,849 bins
  chr12: 669,259 bins
  chr11: 675,032 bins
  chr10: 677,6

In [ ]:
class MixedEmission(dist.TorchDistribution):
    arg_constraints = {
        "bin_probs": constraints.unit_interval,  # Must be [0, 1]
        "con_alpha": constraints.positive,       # Must be > 0
        "con_beta": constraints.positive       # Must be > 0
    }

    def __init__(self, bin_probs, con_alpha, con_beta, obs_mask = None):
        self.bin_probs = bin_probs      # [states, binary_count]
        if con_alpha.dim() == 1:  # Need to make 1D tensor into 2D: each state emits 1D value
            con_alpha = con_alpha.unsqueeze(-1)
        self.con_alpha = con_alpha   # [states, num_cont]
        if con_beta.dim() == 1:
            con_beta = con_beta.unsqueeze(-1)
        self.con_beta  = con_beta # [states, num_cont]
        # bin_probs shape: ( s, binary_count ) -> (states, num_binary_features)
        # con_alpha shape:  (s, con_count)  -> (states, num_continuous_features)
        self.obs_mask = obs_mask
        #obs_mask shape: (states, total features)
        #print(f"bin_probs.shape: {bin_probs.shape}")
        #print(f"con_alpha.shape: {con_alpha.shape}")
        #print(f"con_beta.shape: {con_beta.shape}")

        batch_shape = bin_probs.shape[:-1]
        event_shape = torch.Size([bin_probs.shape[-1] + con_alpha.shape[-1]])

        super().__init__(batch_shape=batch_shape, event_shape=event_shape)
        #super().__init__(batch_shape=torch.Size([2, 2])) #batch, states


        self.bin_dist = dist.Bernoulli(bin_probs).to_event(1) # emissions matrix as input
        self.con_dist = dist.Beta(con_alpha, con_beta).to_event(1) # inputs alpha and beta

    def log_prob(self, value):
        # value shape: (..., 2)
        #print(f"value.shape: {value.shape}")

        num_bin = int(self.bin_probs.shape[-1])
        num_con = int(self.con_alpha.shape[-1])

        # Advanced indexing isn't JIT-friendly
        #bin_data = value[..., :num_bin]
        #con_data = value[..., num_bin:]

        # Assuming value has shape (Batch, Time, num_emissions):
        bin_data = torch.narrow(value, -1, 0, num_bin)
        con_data = torch.narrow(value, -1, num_bin, num_con)

        # Beta distribution undefined for 0 or 1 data
        # Using JIT_safe epsilon based on data type to clamp
        eps = torch.finfo(value.dtype).eps * 10
        con_data_clamped = con_data.clamp(min=eps, max=1.0 - eps)

        bin_log_prob = self.bin_dist.log_prob(bin_data)

        con_log_prob = self.con_dist.log_prob(con_data_clamped)

        if self.obs_mask is not None:
            con_mask = torch.narrow(self.obs_mask.float(), -1, num_bin, num_con ) #extract mask for continuous part and make False =0
            con_log_prob = con_log_prob * con_mask #make all unneeded values 0 so they add nothing to the sum

        # Sum the log probabilities of the binary and continuous parts
        sum_log_prob = bin_log_prob + con_log_prob
        #sum_log_prob = bin_log_prob.sum(dim=-1) + con_log_prob.sum(dim=-1)
        #print(f"log_prob return value: {sum_log_prob.shape}")
        return sum_log_prob

    def sample(self, sample_shape=torch.Size()):
        s_disc = self.bin_dist.sample(sample_shape)
        s_cont = self.con_dist.sample(sample_shape)
        return torch.cat([s_disc, s_cont], dim=-1)

In [ ]:
def model_chipseq(sequences, lengths, eff_features, num_cont_dim = args.num_cont_dim, hidden_dim = args.hidden_dim, jit = args.jit, batch_size=None, include_prior=True):
    with ignore_jit_warnings():
        dev = sequences.device  # Anchor PyTorch device to sequences
        num_sequences, max_length, num_marks = map(int, sequences.shape)
        assert lengths.shape == (num_sequences,)
        assert lengths.max() <= max_length
        num_cont  = num_cont_dim
        num_features = eff_features.shape[-1]
        #print(f"num_seq: {num_sequences}, max_len: {max_length}, num_marks: {num_marks}, num_cont: {num_cont}, num_feat: {num_features}")

        # This assertion ensures you have one length value per sequence
        assert lengths.shape == (num_sequences,)
        # This assertion ensures your length values aren't longer than the actual data
        assert lengths.max() <= max_length

        with poutine.mask(mask=include_prior):
            # Initial state vector prior: uniform
            probs_init = pyro.sample(
                "probs_init",
                dist.Dirichlet(torch.ones(hidden_dim, device=dev)).to_event(0),
            )
            # Transition matrix prior: bias toward self-transitions
            probs_x = pyro.sample(
                "probs_x",
                dist.Dirichlet(0.9 * torch.eye(hidden_dim, device=dev) + 0.1).to_event(1),
            )
            # Emission prior: most marks off in most states
            probs_y = pyro.sample(
                "probs_y",
                dist.Beta(0.1, 0.9).expand([hidden_dim, num_marks-num_cont]).to_event(2),
            )

            #Efficiency prior: alpha & beta for linear weight matrix mapping features to Beta params
            w_mu = pyro.param(
                "w_mu",
                torch.randn(hidden_dim, num_features, device=dev) * 0.1
            )

            b_mu = pyro.param(
                "b_mu",
                torch.zeros(hidden_dim, device=dev)
            )

            # Efficiency prior: concentration, higher phi is lower variance
            phi = pyro.param(
                "phi",
                torch.ones(hidden_dim, device=dev) * 10,
                constraint=dist.constraints.positive
            )
        with pyro.plate("sequences", num_sequences, batch_size, dim=-1) as batch:
            lengths_batch = lengths[batch]
            y = sequences[batch] if jit else sequences[batch, :lengths_batch.max()]
            #y = y.unsqueeze(1)
            #print(f"y: {y.shape}")

            # Computing dynamic alpha & beta for Beta distribution using features
            # Beta mean depends on feature, Beta variance isn't currently
            # Get features for the current batch: [batch, T, num_features]
            eff_features_batch = eff_features[batch, :lengths_batch.max()]
            eff_features_batch = eff_features_batch.nan_to_num(0)  # replace NaN with 0 before matmul
            # Compute the mean (mu) for each state at each time step
            logits = torch.matmul(eff_features_batch, w_mu.t()) + b_mu
            logits = logits.clamp(-10, 10)          # keep μ away from 0/1
            mu = torch.sigmoid(logits)
            # Transform mu and phi into alpha and beta
            mu = mu.unsqueeze(-1)
            phi_expanded = phi.reshape(1, 1, hidden_dim, 1)
            alpha_dynamic = mu * phi_expanded
            beta_dynamic = (1 - mu) * phi_expanded

            # Safety clamp – strictly positive
            eps = torch.finfo(alpha_dynamic.dtype).eps
            alpha_dynamic = alpha_dynamic.clamp(min=eps)
            beta_dynamic  = beta_dynamic.clamp(min=eps)

            # Initial state:
            init_logits = probs_init.log()
            #print(f"init_logits: {init_logits}")

            # Transition logits from learned probs_x
            trans_logits = probs_x.log()  # [hidden_dim, hidden_dim]
            #print(f"model_chipseq trans_logits: {trans_logits}")

            #w_mu = w_mu.unsqueeze(-1)
            #b_mu = b_mu.unsqueeze(-1)
            #print(f"probs_y.shape: {probs_y.shape}")
            #print(f"w_mu.shape: {w_mu.shape}")
            #print(f"b_mu.shape: {b_mu.shape}")
            #print(f"phi.shape: {phi.shape}")
            #print(f"alpha_dynamic.shape: {alpha_dynamic.shape}")
            #create Nan Mask for missing data as an input for the MixedEmissions distribution
            nan_mask = ~torch.isnan(y) if torch.isnan(y).any() else None  #create mask but only if Nan's exist
            y = y.nan_to_num(0)  #turn Nan to 0s for no errors (they will disappear in masking
            #create Nan Mask for missing data as an input for the MixedEmissions distribution
            obs_dist = MixedEmission(
                probs_y,
                alpha_dynamic,
                beta_dynamic,
                nan_mask
            )

            #print(f"Init: {init_logits.shape}")   # Should be [4]
            #print(f"Trans: {trans_logits.shape}") # Should be [4, 4]
            #print(f"obs_dist batch_shape: {obs_dist.batch_shape}, event_shape: {obs_dist.event_shape}")

            hmm_dist = dist.DiscreteHMM(init_logits, trans_logits, obs_dist)
            pyro.sample("y", hmm_dist, obs=y)


In [ ]:
pyro.clear_param_store()

guide = AutoDelta(
    poutine.block(model_chipseq,
                  expose_fn=lambda msg: msg["name"].startswith("probs_"))
)

if args.jit:
    print("Jit in use")
    elbo = JitTrace_ELBO(
        max_plate_nesting=1,
        strict_enumeration_warning=True,
    )
else:
    print("No Jit")
    elbo = Trace_ELBO(
        max_plate_nesting=1,
        strict_enumeration_warning=True,
    )

optim = Adam({"lr": args.learning_rate})
svi = SVI(model_chipseq, guide, optim, elbo)

import pickle
import time

losses = []
for step in range(500):
    t0 = time.time()
    loss = svi.step(sequences=sequences_0, lengths=lengths_0, eff_features=eff_features)
    elapsed = time.time() - t0
    losses.append(loss)

    if step % 10 == 0:
        a = pyro.param("w_mu").data
        p = pyro.param("b_mu").data
        print(f"step: {step:>6}  loss: {loss:>12.2f}  "
              f"{elapsed:.1f}s/step  "
              f"a1: {a[0]}  a2: {a[1]}  "
              f"p1: {p[0]}  p2: {p[1]}")

    # Checkpoint every 50 steps
    if step % 50 == 0 and step > 0:
        checkpoint_path = os.path.join(DRIVE_PATH, f'checkpoint_step{step}.pkl')
        with open(checkpoint_path, 'wb') as f:
            pickle.dump({
                'w_mu':       pyro.param("w_mu").detach().cpu(),
                'b_mu':       pyro.param("b_mu").detach().cpu(),
                'phi':        pyro.param("phi").detach().cpu(),
                'probs_y':    pyro.param("AutoDelta.probs_y").detach().cpu(),
                'probs_x':    pyro.param("AutoDelta.probs_x").detach().cpu(),
                'probs_init': pyro.param("AutoDelta.probs_init").detach().cpu(),
                'step':       step,
                'losses':     losses,
            }, f)
        print(f"  Checkpoint saved at step {step}")

# Final save after all steps complete
import pickle
save_path = os.path.join(DRIVE_PATH, 'trained_params.pkl')
with open(save_path, 'wb') as f:
    pickle.dump({
        'w_mu':        pyro.param("w_mu").detach().cpu(),
        'b_mu':        pyro.param("b_mu").detach().cpu(),
        'phi':         pyro.param("phi").detach().cpu(),
        'probs_y':     pyro.param("AutoDelta.probs_y").detach().cpu(),
        'probs_x':     pyro.param("AutoDelta.probs_x").detach().cpu(),
        'probs_init':  pyro.param("AutoDelta.probs_init").detach().cpu(),
        'chrom_names': chrom_names,
        'args':        vars(args),
        'losses':      losses,
    }, f)
print(f"Saved trained parameters to {save_path}")

No Jit


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.18 GiB. GPU 0 has a total capacity of 14.56 GiB of which 411.81 MiB is free. Including non-PyTorch memory, this process has 14.16 GiB memory in use. Of the allocated memory 12.85 GiB is allocated by PyTorch, and 1.18 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
  Trace Shapes:        
   Param Sites:        
  Sample Sites:        
probs_init dist | 11   
          value | 11   
   probs_x dist | 11 11
          value | 11 11
   probs_y dist | 11  9
          value | 11  9
Trace Shapes:
 Param Sites:
Sample Sites: